In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('2019-Oct_funnel-smartphone.csv')

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart
0,2019-10-01 09:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,False,9,1,Tuesday,오전 (6-12시),False,0,0
1,2019-10-01 09:00:11,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb,False,9,1,Tuesday,오전 (6-12시),False,0,0
2,2019-10-01 09:00:11,view,1005011,2053013555631882655,electronics.smartphone,samsung,900.64,530282093,50a293fb-5940-41b2-baf3-17af0e812101,False,9,1,Tuesday,오전 (6-12시),False,0,0
3,2019-10-01 09:00:19,view,1005135,2053013555631882655,electronics.smartphone,apple,1747.79,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,False,9,1,Tuesday,오전 (6-12시),False,0,0
4,2019-10-01 09:00:20,view,1003306,2053013555631882655,electronics.smartphone,apple,588.77,555446831,6ec635da-ea15-4a5d-96b4-c8ca9d38f89f,False,9,1,Tuesday,오전 (6-12시),False,0,0


In [11]:
df = df.sort_values(['user_session', 'event_time']).copy()

In [12]:
grouped = df.groupby(['user_session', 'product_id'])

In [13]:
event_first_time = (
    df
    .groupby(['user_session', 'product_id', 'event_type'])['event_time']
    .min()
    .unstack()
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time.columns:
        event_first_time[col] = pd.NaT

# 존재 여부 컬럼 따로 생성
event_first_time['has_view'] = event_first_time['view'].notna()
event_first_time['has_cart'] = event_first_time['cart'].notna()
event_first_time['has_purchase'] = event_first_time['purchase'].notna()

# 순서 검증
event_first_time['view_to_cart'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    (event_first_time['view'] < event_first_time['cart'])
)

event_first_time['view_to_cart_to_purchase'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    event_first_time['has_purchase'] &
    (event_first_time['view'] < event_first_time['cart']) &
    (event_first_time['cart'] < event_first_time['purchase'])
)

funnel_df = event_first_time[[
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

funnel_df['segment'] = np.select(
    [
        funnel_df['view_to_cart_to_purchase'],  # 완전 전환
        
        funnel_df['view_to_cart'] & ~funnel_df['view_to_cart_to_purchase'],  # cart까지 갔다가 이탈
        
        funnel_df['view'] & ~funnel_df['view_to_cart']  # view만 하고 이탈
    ],
    [
        'conversion',
        'view_cart_drop',
        'view_only_drop'
    ],
    default='etc'
)

In [14]:
segment_summary = (
    funnel_df['segment']
    .value_counts()
    .reset_index()
)

segment_summary.columns = ['segment', 'count']

segment_summary['ratio'] = (
    segment_summary['count'] / segment_summary['count'].sum() * 100
)

segment_summary

,segment,count,ratio
0,view_only_drop,6694294,94.820109
1,conversion,191430,2.711475
2,view_cart_drop,172935,2.449506
3,etc,1335,0.018909


In [15]:
funnel_df['segment'].value_counts(normalize=True) * 100

segment
view_only_drop    94.820109
conversion         2.711475
view_cart_drop     2.449506
etc                0.018909
Name: proportion, dtype: float64

In [16]:
funnel_summary = pd.DataFrame({
    'step': ['view', 'view_to_cart', 'view_to_cart_to_purchase'],
    'count': [
        funnel_df['view'].sum(),
        funnel_df['view_to_cart'].sum(),
        funnel_df['view_to_cart_to_purchase'].sum()
    ]
})

funnel_summary

,step,count
0,view,7058659
1,view_to_cart,364365
2,view_to_cart_to_purchase,191430


In [17]:
# 1. 구매 전환 케이스
conversion_df = funnel_df[
    funnel_df['segment'] == 'conversion'
].copy()

# 2. view만 하고 이탈한 케이스
view_only_drop_df = funnel_df[
    funnel_df['segment'] == 'view_only_drop'
].copy()

# 3. view → cart 후 구매 없이 이탈한 케이스
view_cart_drop_df = funnel_df[
    funnel_df['segment'] == 'view_cart_drop'
].copy()